# FASE 1 - CÓPIA BRONZE
Criação do dataframe raw **bronze**, que possui os dados ainda sujos.

In [49]:
import pandas as pd

bronze = pd.read_excel("pedidos_metalurgica_mezzalira.xlsx")
display(bronze)

,Pedido,Data,Cliente,Cidade,Produto,Valor (R$)
0,1001.0,2026-01-01,Irrigação Vale Forte,Petrolina,"Tubo galvanizado 2""",3260
1,1002.0,2026-01-02 00:00:00,Grades Caruaru,Caruaru,"Barra chata 1""",3570
2,1003.0,2026-01-06 00:00:00,Serralheria Capibaribe,recife,Telha trapezoidal,1940
3,1004.0,2026-01-07 00:00:00,Serralheria Capibaribe,Recife,Chapa de aço 3 mm,4910
4,1005.0,2026-01-16,Agro Estruturas São Francisco,Petrolina,Perfil U enrijecido,4500
5,1006.0,2026-01-27,Irrigação Vale Forte,Petrolina,"Cantoneira 1/2""",1350
6,1007.0,01-fev-2026,Grades Caruaru,Caruaru,Perfil U enrijecido,5360
7,1008.0,2026-02-04 00:00:00,Estaleiro Porto Novo,REC,"Cantoneira 1/2""",3900
8,1009.0,2026-02-06 00:00:00,Serralheria Capibaribe,Recife,"Barra chata 1""",4040
9,1010.0,2026-02-07 00:00:00,Calhas Olinda,Olinda,"Tubo galvanizado 2""",5700


# FASE 2 - INÍCIO DA LIMPEZA (DF PRATA)
Aqui vamos criar o dataframe **prata** e realizar as limpezas necessárias nos dados nele.

In [50]:
prata = bronze.copy() # Cópia do dataframe
log = [] # Lista que irá armazenar o log de atualizações
display(prata.info()) # Mostra as informações do dataframe

def registro_log(problema, solucao, linhas_afetadas, justificativa):
    log.append({
        "O que encontrei: ": problema,
        "O que fiz: ": solucao,
        "Linhas afetadas: ": int(linhas_afetadas),
        "Justificativa: ": justificativa
    })
    print(f"{problema}.\nSolução: {solucao}. {linhas_afetadas} linha(s) modificada(s).\nMotivo: {justificativa}")

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Pedido      57 non-null     float64
 1   Data        57 non-null     object 
 2   Cliente     57 non-null     str    
 3   Cidade      57 non-null     str    
 4   Produto     57 non-null     str    
 5   Valor (R$)  57 non-null     object 
dtypes: float64(1), object(2), str(3)
memory usage: 2.9+ KB


None

## 1º) Limpeza de Linhas Vazias
Após analisar o que o dataframe prata possui, notamos imediatamente uma inconsistência entre o nº de linhas não nulas e o nº total de linhas da planilha, o que significa que ela possui ao menos três linhas NaN.

In [51]:
linhas_antes = len(prata) # Estado anterior à remoção das linhas
prata = prata.dropna() # Exclui todas as ocorrências de linhas inteiramente vazias

registro_log("Linhas sem dados", "Exclusão das linhas NaN", linhas_antes - len(prata), "Não possuem dados acerca dos pedidos.")

Linhas sem dados.
Solução: Exclusão das linhas NaN. 3 linha(s) modificada(s).
Motivo: Não possuem dados acerca dos pedidos.


## 2º) Análise de Linhas Duplicadas
Em seguida, vamos seguir com a remoção de linhas duplicadas. Como cada pedido é distinguido através do seu ID, linhas que possuam ID de pedidos iguais representam um mesmo pedido. Para isso, precisamos garantir que todos os IDs estão representados de uma mesma maneira (neste caso, um float).

Como pudemos ver acima, ao utilizar o **prata.info()**, obtivemos a informação de que **todos os dados da coluna "Pedidos" estão representados como um valor do tipo float**, sem nenhum tipo de inconsistência que poderia atrapalhar a contagem de duplicatas (caso um dos valores repetidos fosse uma string, por exemplo). Dessa forma, a comparação baseada no ID é justa e confiável.

In [52]:
duplicatas = prata.duplicated(subset=['Pedido']).sum() # Retorna o total de Pedidos que aparecem mais de uma vez

prata = prata.drop_duplicates(subset="Pedido", keep="first") # Mantém a primeira ocorrência e remove as demais linhas duplicadas.
registro_log("Presença de linhas duplicadas", "Remoção das copiadas. Manteve 1º", duplicatas, "Apenas uma linha é necessária para representar cada pedido")

Presença de linhas duplicadas.
Solução: Remoção das copiadas. Manteve 1º. 4 linha(s) modificada(s).
Motivo: Apenas uma linha é necessária para representar cada pedido


## 3º) Correção de Inconsistências de Tipo
Com as linhas repetidas já removidas, vamos agora corrigir inconsistências nos tipos de uma mesma coluna.

De todas as colunas do dataframe, apenas a coluna de **"Valor (R$)"** é do tipo **object**, já que ela possui dados do tipo string e inteiro. Nenhum dos valores possui dígitos após a vírgula, então façamos a conversão de todos para o tipo inteiro.

In [53]:
def converte_inteiro(string): # Funçõa de conversão de string para inteiro
    if isinstance(string, str): # Caso o valor seja do tipo str e não do tipo int
        string = string.replace("R$", "").strip()
        string = string.replace(",00", "")
        string = string.replace(".", "")
    
    return int(string)

qtd_string = prata["Valor (R$)"].str.contains("R$", regex=False).sum() # Conta a quantidade de valores que possuem a string "R$"
prata["Valor (R$)"] = prata["Valor (R$)"].apply(converte_inteiro) # Realiza a conversão

registro_log("Alguns elementos da coluna 'Valores (R$)' são representados como string", "Mudança de tipo para inteiro", qtd_string, "O tipo inteiro/float facilita a comparação de magnitude entre os preços, e permite a realização de operações entre os valores obtidos")

Alguns elementos da coluna 'Valores (R$)' são representados como string.
Solução: Mudança de tipo para inteiro. 7 linha(s) modificada(s).
Motivo: O tipo inteiro/float facilita a comparação de magnitude entre os preços, e permite a realização de operações entre os valores obtidos


## 4º) Padronização de Representação
Um dos problemas encontrados no dataframe é a representação distinta da cidade Recife, com mais de uma sigla distinta. É necessário substituí-las para um valor em comum para que os dados estejam agrupados.

Além disso, o modelo das datas também é apresentado diferente. Para facilitar a leitura, vamos utilizar um único modelo.

In [54]:
prata["Cidade"].value_counts().reset_index() # Conta o nº de vezes que cada elemento da coluna "Cidade" aparece e mostra em uma tabela

,Cidade,count
0,Caruaru,11
1,Olinda,9
2,Jaboatão dos Guararapes,9
3,Petrolina,8
4,Recife,6
5,recife,5
6,REC,5


É possível observar que há três representações distintas para a cidade Recife, e todas precisam se mesclar para uma representação única.

In [55]:
correcao = { # Dicionário de correções
    "recife": "Recife",
    "REC": "Recife",
}

antes = prata["Cidade"].copy() # Copia a coluna de "Cidade"

prata["Cidade"] = prata["Cidade"].replace(correcao) # Aplica as correções do dicionário

qtd_correcoes = (antes != prata["Cidade"]).sum() # Conta a quantidade de linhas que foram modificadas.
registro_log("Múltiplas representações para Recife", "Padronização para o modelo 'Recife'", qtd_correcoes, "Agora que todas as representações das cidades estão no mesmo modelo, a análise do desempenho de cada cidade é mais direta")

prata["Cidade"].value_counts().reset_index()

Múltiplas representações para Recife.
Solução: Padronização para o modelo 'Recife'. 10 linha(s) modificada(s).
Motivo: Agora que todas as representações das cidades estão no mesmo modelo, a análise do desempenho de cada cidade é mais direta


,Cidade,count
0,Recife,16
1,Caruaru,11
2,Olinda,9
3,Jaboatão dos Guararapes,9
4,Petrolina,8


Agora, resta padronizar a representação das datas dispostas na coluna "Data".

In [56]:
meses = {"jan": "01", "fev": "02", "mar": "03", "abr": "04", "mai": "05", "jun": "06",
         "jul": "07", "ago": "08", "set": "09", "out": "10", "nov": "11", "dez": "12"}

datas = prata["Data"].replace(meses, regex=True) # Substitui as ocorrências dos meses por extenso

# Pega cada um dos formatos presentes no dataframe e tenta realizar a substituição para datetime
f_iso = pd.to_datetime(datas, format="%Y-%m-%d", errors="coerce")
f_tracinho = pd.to_datetime(datas, format="%d-%m-%Y", errors="coerce")
f_hora = pd.to_datetime(datas, format="%Y-%m-%d %H:%M:%S", errors="coerce")

fora_padrao = f_iso.isna().sum() # Todas as datas que não estiverem no formato ISO são consideradas fora do padrão

prata["Data"] = f_iso.fillna(f_tracinho).fillna(f_hora) # Substituição das datas

registro_log("Presença de datas em formatos diferentes", "Conversão das linhas fora do formato datetime na representação correta", fora_padrao, "Com todas as datas representadas de maneira equivalente, a comparação é direta e rápida, e permite melhor análise temporal entre os pedidos")

print("\nCamada Prata Concluída:")
display(prata.info())
display(prata)

Presença de datas em formatos diferentes.
Solução: Conversão das linhas fora do formato datetime na representação correta. 6 linha(s) modificada(s).
Motivo: Com todas as datas representadas de maneira equivalente, a comparação é direta e rápida, e permite melhor análise temporal entre os pedidos

Camada Prata Concluída:
<class 'pandas.DataFrame'>
Index: 53 entries, 0 to 58
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   Pedido      53 non-null     float64       
 1   Data        53 non-null     datetime64[us]
 2   Cliente     53 non-null     str           
 3   Cidade      53 non-null     str           
 4   Produto     53 non-null     str           
 5   Valor (R$)  53 non-null     int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 2.9 KB


None

,Pedido,Data,Cliente,Cidade,Produto,Valor (R$)
0,1001.0,2026-01-01,Irrigação Vale Forte,Petrolina,"Tubo galvanizado 2""",3260
1,1002.0,2026-01-02,Grades Caruaru,Caruaru,"Barra chata 1""",3570
2,1003.0,2026-01-06,Serralheria Capibaribe,Recife,Telha trapezoidal,1940
3,1004.0,2026-01-07,Serralheria Capibaribe,Recife,Chapa de aço 3 mm,4910
4,1005.0,2026-01-16,Agro Estruturas São Francisco,Petrolina,Perfil U enrijecido,4500
5,1006.0,2026-01-27,Irrigação Vale Forte,Petrolina,"Cantoneira 1/2""",1350
6,1007.0,2026-02-01,Grades Caruaru,Caruaru,Perfil U enrijecido,5360
7,1008.0,2026-02-04,Estaleiro Porto Novo,Recife,"Cantoneira 1/2""",3900
8,1009.0,2026-02-06,Serralheria Capibaribe,Recife,"Barra chata 1""",4040
9,1010.0,2026-02-07,Calhas Olinda,Olinda,"Tubo galvanizado 2""",5700


## LOG DE CORREÇÕES

In [57]:
df_log = pd.DataFrame(log)
display(df_log)

,O que encontrei:,O que fiz:,Linhas afetadas:,Justificativa:
0,Linhas sem dados,Exclusão das linhas NaN,3,Não possuem dados acerca dos pedidos.
1,Presença de linhas duplicadas,Remoção das copiadas. Manteve 1º,4,Apenas uma linha é necessária para representar...
2,Alguns elementos da coluna 'Valores (R$)' são ...,Mudança de tipo para inteiro,7,O tipo inteiro/float facilita a comparação de ...
3,Múltiplas representações para Recife,Padronização para o modelo 'Recife',10,Agora que todas as representações das cidades ...
4,Presença de datas em formatos diferentes,Conversão das linhas fora do formato datetime ...,6,Com todas as datas representadas de maneira eq...


# FASE 3 - CAMADA OURO
Agora que a limpeza dos dados está concluída, podemos ingressar na análise do que foi coletado.

A pergunta que desejamos responder é **"quais cidades venderam mais em cada mês?"**

In [58]:
base = prata.assign(Mes=prata["Data"].dt.month) # Cópia com uma coluna extra representando o mês

ouro = base.pivot_table( # Cria a tabela Ouro, com as informações necessárias
    index="Cidade",         # Nome da cidade (representam as linhas)
    columns="Mes",          # Qual o mês de faturamento (representa as colunas)
    values="Valor (R$)",    # Os argumntos que serão somados
    aggfunc="sum",          # A operação que vai ser feita
    fill_value=0,           # Começa do 0 caso o faturamento seja 0
    margins=True,           # Adiciona os valores totais
    margins_name="Venda Total", # Nome da coluna com os totais
)

n_meses = {1: "Jan", 2: "Fev", 3: "Mar", 4: "Abr", 5: "Maio", 6: "Jun", 
           7: "Jul", 8: "Ago", 9: "Set", 10: "Out", 11: "Nov", 12: "Dez"}

ouro = ouro.rename(columns=n_meses) # Renomeia os meses em números de acordo com o valor por extenso
display(ouro)

Mes,Jan,Fev,Mar,Abr,Venda Total
Cidade,,,,,
Caruaru,3570,15030,17610,6930,43140
Jaboatão dos Guararapes,0,3920,22820,1120,27860
Olinda,0,10100,7710,11190,29000
Petrolina,9110,8770,1590,3610,23080
Recife,6850,16240,29730,2440,55260
Venda Total,19530,54060,79460,25290,178340


Por fim, vamos organizar as informações obtidas e responder a pergunta.

In [59]:
cidades = ouro.drop(index="Venda Total") # Remove a linha de venda total, para comparar somente as cidades.

melhor_cidade = cidades["Venda Total"].idxmax() # Pega a cidade com maior faturamento

melhor_mes = cidades.drop(columns="Venda Total").loc[melhor_cidade].idxmax() # Retorna o mês que a cidade com maior faturamento vendeu mais

print(f"A cidade que mais comprou foi {melhor_cidade}, com o total de R${cidades["Venda Total"].idxmax()},00.")
print(f"O mês de pico de compras de {melhor_cidade} foi {melhor_mes}.")

A cidade que mais comprou foi Recife, com o total de R$Recife,00.
O mês de pico de compras de Recife foi Mar.


# Portanto...
Chegamos à conclusão de que **Recife** foi a cidade que mais realizou compras, e que o mês com maior tráfego de compras na cidade foi **Março**.

Abaixo são os arquivos para download de cada uma das etapas.

In [60]:
etapa_bronze = "etapa_bronze.csv"
bronze.to_csv(etapa_bronze, index="False")

etapa_prata = "etapa_prata.csv"
prata.to_csv(etapa_prata, index="False")

log_atualizado = "log_atualizado.csv"
df_log.to_csv(log_atualizado, index="False")

etapa_ouro = "etapa_ouro.csv"
ouro.to_csv(etapa_ouro, index="False")